In [ ]:

import pdfplumber, re

BOILER = re.compile(r"Certified WCAG 2\.1 Level AA compliant.*?SWH\s*\d*")

def clean(text):
    text = BOILER.sub("", text)
    return re.sub(r"[ \t]+\n", "\n", text).strip()

def table_to_rows(table):
    header = [(c or "").replace("\n", " ").strip() for c in table[0]]
    rows = []
    for row in table[1:]:
        cells = [(c or "").replace("\n", " ").strip() for c in row]
        parts = [f"{h}: {c}" for h, c in zip(header, cells) if c]
        if parts:
            rows.append("; ".join(parts))
    return rows

def extract(path, start_page=3):
    items = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            if i < start_page:
                continue
            text = clean(page.extract_text() or "")
            if text:
                items.append({"page": i, "type": "text", "text": text})
    return items

items = extract("sustainability-action-plan.pdf")

items = extract("sustainability-action-plan.pdf")
print(len(items))

In [24]:
print(len(items))
print(items[0])

57
{'page': 3, 'type': 'text', 'text': 'City of Winter Park\nSustainability Action Plan\nJanuary 24th, 2024\nPresented by:\nGloria Eby, Natural Resources and Sustainability Director\nSara Miller, Sustainability Manager\nMia Brady, Sustainability Specialist\nKeep Winter Park Beautiful & Sustainable Advisory Board'}


In [25]:
from collections import Counter

print(Counter(i["type"] for i in items))
per_page = Counter(i["page"] for i in items)
print(sorted(per_page.items()))

Counter({'text': 57})
[(3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 1), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 1)]


In [26]:
print(sum("WCAG" in i["text"] for i in items))   # want 0

0


In [27]:
lengths = sorted(items, key=lambda i: len(i["text"]))
for i in lengths[:5]:  print(i["page"], repr(i["text"]))   # shortest
for i in lengths[-3:]: print(i["page"], len(i["text"]))    # longest

3 'City of Winter Park\nSustainability Action Plan\nJanuary 24th, 2024\nPresented by:\nGloria Eby, Natural Resources and Sustainability Director\nSara Miller, Sustainability Manager\nMia Brady, Sustainability Specialist\nKeep Winter Park Beautiful & Sustainable Advisory Board'
41 'Projected Responsible\nImplementation Action Accomplished Status Department/\nYear Division\nAdminister integrated aquatic plant\nContinue Annually Yes Ongoing Lakes Division\nmanagement program\nProvide Tree Canopy Conservation\neducation (e.g., environmental/health Urban Forestry,\nbenefits, cost savings, aesthetics) to Sustainability\nContinue Annually residents, building professionals, realtors Yes Ongoing Division, Building\nand businesses through on-line, social & Permitting,\nmedia, print campaigns and in person Communications\nworkshops\nNatural\nResources &\nIncrease Watershed Clean Ups, Tree\nContinue Annually Yes Ongoing Sustainability\nPlantings, and Shore Restoration Events\nDept., Urban\nForestr

In [28]:
for i in items:
    if i["type"] == "table_row" and 10 <= i["page"] <= 13:
        print(i["page"], i["text"], "\n")

In [29]:
def clean_cell(c):
    return (c or "").replace("\n", " ").strip()

def looks_like_header(row):
    cells = [clean_cell(c) for c in row if clean_cell(c)]
    if not cells:
        return False
    numeric = sum(bool(re.fullmatch(r"[-–\d.,%$\s]+", c)) for c in cells)
    return numeric / len(cells) < 0.3

last_header = {}   # column count -> most recent real header

def table_to_rows(data):
    ncols = len(data[0])
    if looks_like_header(data[0]):
        header = [clean_cell(c) for c in data[0]]
        last_header[ncols] = header
        body = data[1:]
    else:
        header = last_header.get(ncols)   # may be None
        body = data

    rows = []
    for row in body:
        cells = [clean_cell(c) for c in row]
        if header:
            parts = [f"{h}: {c}" for h, c in zip(header, cells) if c]
            text = "; ".join(parts)
        else:
            text = " | ".join(c for c in cells if c)   # unlabeled, but not mislabeled
        if text:
            rows.append(text)
    return rows

In [30]:
with pdfplumber.open("sustainability-action-plan.pdf") as pdf:
    page = pdf.pages[12]              # PDF page 13
    for t in page.find_tables():
        for row in t.extract()[:3]:
            print(row)

['Proportion of renewable energy in\nWinter Park Electric Utility’s\nEnergy portfolio1 – Baseline Year:\n2021 includes Covanta', '4%', '13%', '21.72', '23%', '80%', '89%', None, None, None]
['Community wide greenhouse gas\nemissions [Tons of carbon dioxide\nequivalent]– Baseline Year: 2018', '398,919', '405,394', '406,999', '355,000', '235,000', '151,000', None, None, None]
['WP Electric Utility customers with\nSolar – Baseline Year: 2012', '7', '139', '154', '300', '700', '950', None, None, None]


In [33]:
def find(term, items=items, width=250):
    for i in items:
        text = i["text"]
        pos = text.lower().find(term.lower())
        if pos != -1:
            start = max(0, pos - width)
            print(f'page {i["page"]} ({i["type"]}):')
            print(text[start : pos + width], "\n")

find("CR-1")
find("58,000")

page 13 (text):
n
is presented in all subsequent tables labeled as “2021 Status”.
INDICATORS
2012 2021 2022 2025 2035 2042 2050
Indicator Description
Baseline Status Status Target Target Target Target
Proportion of renewable energy in
Winter Park Electric Utility’s
CR-1 4% 13% 21.72 23% 80% 89% 100%
Energy portfolio1 – Baseline Year:
2021 includes Covanta
Community wide greenhouse gas
CR-2 emissions [Tons of carbon dioxide 398,919 405,394 406,999 355,000 235,000 151,000 58,000
equivalent]– Baseline Year: 2018
W 

page 13 (text):
energy in
Winter Park Electric Utility’s
CR-1 4% 13% 21.72 23% 80% 89% 100%
Energy portfolio1 – Baseline Year:
2021 includes Covanta
Community wide greenhouse gas
CR-2 emissions [Tons of carbon dioxide 398,919 405,394 406,999 355,000 235,000 151,000 58,000
equivalent]– Baseline Year: 2018
WP Electric Utility customers with
CR-3 7 139 154 300 700 950 1300
Solar – Baseline Year: 2012
Proportion of Residents within 1/2
CR-4 mile of affordable, healthy food - 46% 4

In [43]:
def chunk_items(items, chunk_size=512, overlap=0):
    chunks= []
    
    step = chunk_size - overlap

    for item in items:
        text = item["text"]
        for start in range(0, len(text), step):
            chunk = text[start : start + chunk_size]
            if chunk.strip():
                chunks.append({"page": item["page"], "text": chunk})

    return chunks

    

In [44]:
fixed= chunk_items(items,chunk_size=512,overlap=0)
overlapped= chunk_items(items,chunk_size=512,overlap=100)
print(len(fixed), len(overlapped))
print(fixed[5])

242 294
{'page': 4, 'text': '........................................ 33\nTransportation and Urban Form .................................................................................................. 40\nWaste Management ......................................................................................................................... 43\n2022 Action Item Tracker List: ..................................................................................................... 46\nGlossary .................................................'}


In [45]:
sizes = [len(c["text"]) for c in fixed]
print(max(sizes), min(sizes), sum(sizes)/len(sizes))

512 6 449.004132231405


In [46]:
for name, chunks in [("fixed", fixed),("overlapped", overlapped)]:
    print("=====", name)
    for c in chunks:
        if "CR-3" in c["text"]:
            print(c["page"], "\n", c["text"], "\n")

===== fixed
13 
 rbon dioxide 398,919 405,394 406,999 355,000 235,000 151,000 58,000
equivalent]– Baseline Year: 2018
WP Electric Utility customers with
CR-3 7 139 154 300 700 950 1300
Solar – Baseline Year: 2012
Proportion of Residents within 1/2
CR-4 mile of affordable, healthy food - 46% 40% 50% 75% 80% TBD
options – Baseline Year: 2012
Undergrounding of Winter Park’s
CR-5 - - 73.1% 85% 100%
electrical Utility2
1Approximations with plans for refinement as reaching 100% target in 2050 is contingent on cost reduction in di 

===== overlapped
13 
 035 2042 2050
Indicator Description
Baseline Status Status Target Target Target Target
Proportion of renewable energy in
Winter Park Electric Utility’s
CR-1 4% 13% 21.72 23% 80% 89% 100%
Energy portfolio1 – Baseline Year:
2021 includes Covanta
Community wide greenhouse gas
CR-2 emissions [Tons of carbon dioxide 398,919 405,394 406,999 355,000 235,000 151,000 58,000
equivalent]– Baseline Year: 2018
WP Electric Utility customers with
CR-3 7 139

fixed size and overlap chunking both split the header from the table rows. overlap of 100 was too short to bridge the gap

In [48]:
big_chunk = chunk_items(items, chunk_size=1024, overlap=300)
for c in big_chunk:
    if "CR-3" in c["text"]:
        print(c["page"], "\n", c["text"], "\n")


13 
 rtfolio and opportunities for reducing their carbon footprint
8. Benchmark all indicators to current state of the city given pandemic and
operational delays
The Department of Natural Resources and Sustainability conducted a comprehensive
inventory of all SAP metric indicators to benchmark city status in effort to give the new
Department a starting point on data trend and analysis moving forward. This information
is presented in all subsequent tables labeled as “2021 Status”.
INDICATORS
2012 2021 2022 2025 2035 2042 2050
Indicator Description
Baseline Status Status Target Target Target Target
Proportion of renewable energy in
Winter Park Electric Utility’s
CR-1 4% 13% 21.72 23% 80% 89% 100%
Energy portfolio1 – Baseline Year:
2021 includes Covanta
Community wide greenhouse gas
CR-2 emissions [Tons of carbon dioxide 398,919 405,394 406,999 355,000 235,000 151,000 58,000
equivalent]– Baseline Year: 2018
WP Electric Utility customers with
CR-3 7 139 154 300 700 950 1300
Solar – Baselin